### Modelo de Regresion

***Nombre***: Gerard Almanzar
***Curso***: AI & ML
***Asignatura***: Modelos Supervisados
***Actividad Integradora Un dataset, dos preguntas***

Acerca del dataset: 344 pingüinos de 3 especies, medidos en 3 islas de la Antártida. Cada fila es un pingüino real, medido por investigadores en la Antártida: su especie, la isla donde vive, el largo y el ancho de su pico, el largo de su aleta, su peso corporal, y su sexo.


***Objetivos de la Practica***: utilizar el mismo dataset para entrenar un modelo de **Regresion** (Tarea A) y un modelo de **Clasificacion** (Tarea B), y se debe decidir, en cada caso, que tecnica y que metricas corresponden.





### Antes de programar: decide qué tipo de problema es Con los mismos datos de pingüinos, estas son 6 preguntas de negocio distintas que alguien podría hacer.

Para cada pregunta, marca si necesitarías un modelo de regresión o de clasificación para responderla. No hace falta escribir código todavía. El objetivo es que reconozcan el tipo de problema con solo leer la pregunta, la misma habilidad que ya practicaron en el Mes 1.


- **Cuanto va a pesar un pinguino cuando sea adulto?** Esta pregunta como esta buscando una respuesta numerica se debe de usar una regresion.
- **Es este piguino macho o hembra?** la respuesta se encontraria con un modelo de clasificacion
- **De que especie es este pinguino: Adelie, Gentoo o Chinstrap?** la respuesta se encontraria con un modelo de clasificacion
- **Que tan largo v aa ser el pico de este pinguino?** Esta pregunta como esta buscando una respuesta numerica se debe de usar una regresion.
- **Este Pinquino vive en la isla Biscoe, Dream o Torgersen?** la respuesta se encontraria con un modelo de clasificacion
- **Cuanto mide la aleta de un pinguino con estas caracteristicas?** Esta pregunta como esta buscando una respuesta numerica se debe de usar una regresion.



### Importar librerias requeridas

In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

### Carga del Dataset y Vista preliminar de datos

In [10]:
df = pd.read_csv('../datasets/penguins.csv')

In [11]:
print("Vista preliminar del dataset: ")
print("Dimension del dataset: ", df.shape)
print("\nColumnas del dataset:")
print(list(df.columns))
print("\nTipos de datos del dataset:")
print(df.dtypes)
print("\nPrimeras filas del dataset: ")
print(df.head(10).to_string())
print('\nValores Nulos:')
print(df.isnull().sum())

Vista preliminar del dataset: 
Dimension del dataset:  (344, 7)

Columnas del dataset:
['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']

Tipos de datos del dataset:
species                  str
island                   str
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                      str
dtype: object

Primeras filas del dataset: 
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g     sex
0  Adelie  Torgersen            39.1           18.7              181.0       3750.0    MALE
1  Adelie  Torgersen            39.5           17.4              186.0       3800.0  FEMALE
2  Adelie  Torgersen            40.3           18.0              195.0       3250.0  FEMALE
3  Adelie  Torgersen             NaN            NaN                NaN          NaN     NaN
4  Adelie  Torgersen            36.7           19.3              193.0       

### Regresión: predecir el peso del pingüino


In [12]:
#1 Eliminar las filas con nulos en las columnas que se van a utilizar

df.dropna(subset=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex'], inplace=True)

print('\nValores Nulos DESPUES del cambio:')
print(df.isnull().sum())



Valores Nulos DESPUES del cambio:
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


In [27]:
#2 Codificar las columnas de texto con get_dummies

columnas_categoricas = df.select_dtypes(include=['str','object']).columns
columnas_numericas = df.select_dtypes(include=['int', 'float']).columns

df_enc = pd.get_dummies(df, columnas_categoricas, drop_first=False)

#3 Separar X e y
X = df_enc.drop(columns=['body_mass_g'])
y = df['body_mass_g']

#4 Entrenar, predecir y guardar cada metrica en su propia variable.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

modelo_reg = LinearRegression()
modelo_reg.fit(X_train, y_train)
y_pred = modelo_reg.predict(X_test)

#Guarda cada metrica en una variable
mae_reg = mean_absolute_error(y_test, y_pred)
r2_reg = r2_score(y_test, y_pred)

print("\nResultados de las metricas: ")
print("MAE", round(mae_reg, 2))
print("R2", round(r2_reg, 4))



columnas_categoricas: Index(['species', 'island', 'sex'], dtype='str')
columnas_numericas: Index(['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'], dtype='str')

Resultados de las metricas: 
MAE 196.21
R2 0.8962


**Interpretacion de los resultados**

Es 196 gramos de error un margen pequeño o grande? Basado con el rango del peso minimo (2700) de la variable **body_mass_g** y el peso maximo (6300), un margen de error de 196 gramos representa un margen pequeño.

### Clasificación: predecir el sexo del pingüino

¿Es este pingüino macho o hembra?

In [36]:
df_clf = df.dropna(subset=['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g','sex','species','island'])

df_clf_enc = pd.get_dummies(df_clf, columns=['species', 'island'], drop_first=False)
df_clf_enc['Sex_binario'] = (df_clf_enc['sex'] == 'MALE').astype(int)

X = df_clf_enc.drop(columns=['sex', 'Sex_binario'])
y = df_clf_enc['Sex_binario']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

#Escalar SOLO las columnas numericas, ajustando solo con train
scaler = StandardScaler()
X_train[columnas_numericas] = scaler.fit_transform(X_train[columnas_numericas])
X_test[columnas_numericas] = scaler.transform(X_test[columnas_numericas])

#Entrenar, predecir y evaluar

modelo_clf = LogisticRegression()
modelo_clf.fit(X_train, y_train)
y_pred_clf = modelo_clf.predict(X_test)

#Guardar cada metrica en una variable

acc_clf = accuracy_score(y_test, y_pred_clf)
prec_clf = precision_score(y_test, y_pred_clf)
rec_clf = recall_score(y_test, y_pred_clf)
f1_clf = f1_score(y_test, y_pred_clf)

print("Resultados de las metricas: ")
print("Accuracy:", round(acc_clf, 4))
print("Precision:", round(prec_clf, 4))
print("Recall:", round(rec_clf, 4))
print("F1:", round(f1_clf, 4))
print("\nMatrix de Confusion:")
print(confusion_matrix(y_test, y_pred_clf))

Resultados de las metricas: 
Accuracy: 0.8955
Precision: 0.871
Recall: 0.9
F1: 0.8852

Matrix de Confusion:
[[33  4]
 [ 3 27]]


### Ambos resultados de las Tareas A & B 

In [38]:
print("TAREA A: REGRESION (peso del pinguino): ")
print("MAE:", round(mae_reg, 4), "| R2:", round(r2_reg, 4))
print()
print("TAREA B: CLASIFICACION (sexo del pinguino):")
print("Accuracy:", round(acc_clf, 4), "| Precision:", round(prec_clf, 4), "| Recall:", round(rec_clf, 4))

TAREA A: REGRESION (peso del pinguino): 
MAE: 196.2089 | R2: 0.8962

TAREA B: CLASIFICACION (sexo del pinguino):
Accuracy: 0.8955 | Precision: 0.871 | Recall: 0.9


### Preguntas de cierre

- De las 6 preguntas del Paso 1, elijan una que no hayan resuelto hoy (por ejemplo, la especie del pingüino). ¿Qué tipo de modelo necesitarían, y qué métricas usarían
para evaluarlo?

    Respuesta: para predecir la especie del pinguino se necesitaria utilizar un modelo de clasificacion ya que la variable a predecir seria **species** y no una variable numerica. Las metricas que se usarian serian ['Accuracy', 'Precision', 'Recall', 'F1-Score'].
- El modelo de regresión tuvo R²=0.90; el de clasificación, accuracy=0.90. Si un compañero les dice "ambos modelos son igual de buenos porque los dos dieron 0.90",
¿qué le responderían?

    Respuesta: ambos modelos predicen datos totalmente diferentes por lo que no son igual los modelos. Uno predice el peso (valor numerica) del piguino y el otro modelo predice el sexo (variable categorica nominal) e incluso las metricas que usan para evaluar ambos modelos son diferentes.
- ¿Qué tendrían que cambiar en su código para usar este mismo dataset y predecir la especie del pingüino en vez del sexo? Nómbrenlo en general, no hace falta
escribirlo.

    Respuesta: para poder predecir la especie del pinguino en lugar del sexo se deberia de hacer cambios en la variable X donde en vez de sacar a 'sex', se debera de sacar 'species' del dataset que 'X' utilizaria para entrenar el modelo y a su vez en la variable 'y' se cambiaria por igual a 'sex' por 'species' y ya todo lo demas deberia ser relativamente parecido en el codigo del modelo de clasificacion.